In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from src.classifier.data import CATEGORIES, clean_text
from src.classifier.split import load_split, apply_split
from src.classifier.metrics import per_class_report

SEED = 42
df = pd.read_csv("data/opp115_segments.csv")
df["text"] = df["text"].map(clean_text)
parts = apply_split(df, load_split())

In [2]:
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform(parts["train"]["text"])
Xva = vec.transform(parts["val"]["text"])
ytr = parts["train"][CATEGORIES].values
yva = parts["val"][CATEGORIES].values

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
)
clf.fit(Xtr, ytr)
pred = clf.predict(Xva)

report = per_class_report(yva, pred, CATEGORIES)
report

,precision,recall,f1,support
First Party Collection/Use,0.775,0.836,0.804,177.0
Third Party Sharing/Collection,0.807,0.859,0.832,156.0
User Choice/Control,0.574,0.812,0.672,48.0
"User Access, Edit and Deletion",0.680,0.680,0.680,25.0
Data Retention,0.750,0.429,0.545,14.0
Data Security,0.759,0.815,0.786,27.0
Policy Change,0.750,0.857,0.800,21.0
Do Not Track,1.000,0.833,0.909,6.0
International and Specific Audiences,0.911,0.774,0.837,53.0
Other,0.809,0.726,0.765,157.0


In [3]:
os.makedirs("results", exist_ok=True)
os.makedirs("artifacts/classifier", exist_ok=True)
report.to_csv("results/e4_baseline_val.csv")
joblib.dump({"vec": vec, "clf": clf}, "artifacts/classifier/baseline.joblib")

['artifacts/classifier/baseline.joblib']